In [25]:
SURFACE_ALPHA = 0.18

GRID_GLOW_WIDTH = 1.35
GRID_CORE_WIDTH = 0.48

GRID_GLOW_ALPHA = 0.65
GRID_CORE_ALPHA = 0.98

COL = "#35f6ff"
COL_GRID_U = "white"
COL_GRID_V = "#9ffcff"

# 1. The Klein Bottle

In [16]:
# Klein Bottle — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/klein_bottle_v2_grid.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W, H = 10, 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"

COL = "#35f6ff"
COL_GRID_U = "white"
COL_GRID_V = "#9ffcff"

SURFACE_ALPHA = 0.16
GRID_GLOW_ALPHA = 0.65
GRID_CORE_ALPHA = 0.98

GRID_GLOW_WIDTH = 1.35
GRID_CORE_WIDTH = 0.48

GRID_STEP_U = 6
GRID_STEP_V = 6


# -----------------------------------------------------------------------------
# Klein bottle geometry
# -----------------------------------------------------------------------------

u = np.linspace(0, 2 * np.pi, 220)
v = np.linspace(0, 2 * np.pi, 120)

U, V = np.meshgrid(u, v)

X = np.where(
    U < np.pi,
    3 * np.cos(U) * (1 + np.sin(U))
    + (2 * (1 - np.cos(U) / 2)) * np.cos(U) * np.cos(V),
    3 * np.cos(U) * (1 + np.sin(U))
    + (2 * (1 - np.cos(U) / 2)) * np.cos(V + np.pi),
)

Y = np.where(
    U < np.pi,
    8 * np.sin(U)
    + (2 * (1 - np.cos(U) / 2)) * np.sin(U) * np.cos(V),
    8 * np.sin(U),
)

Z = (2 * (1 - np.cos(U) / 2)) * np.sin(V)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.3
Y = Y / scale * 4.3
Z = Z / scale * 4.3


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.75, 3.75)
    ax.set_ylim(-3.75, 3.75)
    ax.set_zlim(-3.75, 3.75)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    # U-lines
    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    # V-lines
    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=25 + 7 * np.sin(tau * t),
        azim=360 * t,
    )

    colors = build_colors(pulse)

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=colors,
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


# -----------------------------------------------------------------------------
# Export
# -----------------------------------------------------------------------------

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="klein_bottle_v2_grid",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/klein_bottle_v2_grid.webm
Frames: 192
Size: 3504.8 KB


The Klein bottle is a non-orientable surface with no boundary. Unlike an ordinary bottle, it has no inside or outside. In four-dimensional space it can exist without self-intersection, but any three-dimensional representation must pass through itself. It is one of the fundamental objects of topology and can be thought of as a higher-dimensional cousin of the Möbius strip.

# Бутылка Клейна

Бутылка Клейна — знаменитая неориентируемая поверхность, у которой нет ни внутренней, ни внешней стороны. В четырёхмерном пространстве она не пересекает сама себя, но в трёхмерном мире её можно изобразить только с самопересечением.

Параметрическое представление одной из распространённых версий:

$$ \begin{aligned}
x &= \left(r+\cos\frac{u}{2}\sin v-\sin\frac{u}{2}\sin 2v\right)\cos u\\
y &= \left(r+\cos\frac{u}{2}\sin v-\sin\frac{u}{2}\sin 2v\right)\sin u\\
z &= \sin\frac{u}{2}\sin v+\cos\frac{u}{2}\sin 2v
\end{aligned} $$

$$ \begin{aligned}x&=\left(r+\cos\frac{u}{2}\sin v-\sin\frac{u}{2}\sin 2v\right)\cos u\\y&=\left(r+\cos\frac{u}{2}\sin v-\sin\frac{u}{2}\sin 2v\right)\sin u\\z&=\sin\frac{u}{2}\sin v+\cos\frac{u}{2}\sin 2v\end{aligned} $$

Обычно для визуализации используют $r \approx 2$.

# 2. The Schwarz P surface

In [14]:
# Schwarz P Surface — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/schwarz_p_surface_v2_grid.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from skimage import measure
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from vizlib.animation_export import export_animation


OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W, H = 10, 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"

COL = "#35f6ff"

SURFACE_ALPHA = 0.10
EDGE_GLOW_ALPHA = 0.42
EDGE_CORE_ALPHA = 0.88

EDGE_GLOW_WIDTH = 0.42
EDGE_CORE_WIDTH = 0.08

N = 48
L = 2.2 * np.pi


# -----------------------------------------------------------------------------
# Schwarz P surface geometry
# Implicit equation:
#   cos(x) + cos(y) + cos(z) = 0
# -----------------------------------------------------------------------------

x = np.linspace(-L, L, N)
y = np.linspace(-L, L, N)
z = np.linspace(-L, L, N)

X, Y, Z = np.meshgrid(x, y, z, indexing="ij")

F = np.cos(X) + np.cos(Y) + np.cos(Z)

verts, faces, normals, values = measure.marching_cubes(
    F,
    level=0.0,
    spacing=(
        (x.max() - x.min()) / (N - 1),
        (y.max() - y.min()) / (N - 1),
        (z.max() - z.min()) / (N - 1),
    ),
)

verts[:, 0] -= verts[:, 0].mean()
verts[:, 1] -= verts[:, 1].mean()
verts[:, 2] -= verts[:, 2].mean()

verts /= np.max(np.abs(verts)) / 4.15

mesh_faces = verts[faces]


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.25, 3.25)
    ax.set_ylim(-3.25, 3.25)
    ax.set_zlim(-3.25, 3.25)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def make_surface_layer(pulse: float) -> Poly3DCollection:
    alpha = SURFACE_ALPHA * (0.82 + 0.18 * pulse)

    return Poly3DCollection(
        mesh_faces,
        facecolor=(base_rgb[0], base_rgb[1], base_rgb[2], alpha),
        edgecolor=(0, 0, 0, 0),
        linewidth=0.0,
        alpha=alpha,
    )


def make_edge_glow_layer(pulse: float) -> Poly3DCollection:
    alpha = EDGE_GLOW_ALPHA * (0.75 + 0.25 * pulse)

    return Poly3DCollection(
        mesh_faces,
        facecolor=(0, 0, 0, 0),
        edgecolor=(base_rgb[0], base_rgb[1], base_rgb[2], alpha),
        linewidth=EDGE_GLOW_WIDTH,
        alpha=1.0,
    )


def make_edge_core_layer(pulse: float) -> Poly3DCollection:
    alpha = EDGE_CORE_ALPHA * (0.82 + 0.18 * pulse)

    return Poly3DCollection(
        mesh_faces,
        facecolor=(0, 0, 0, 0),
        edgecolor=(1.0, 1.0, 1.0, alpha),
        linewidth=EDGE_CORE_WIDTH,
        alpha=1.0,
    )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=30 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.add_collection3d(make_surface_layer(pulse))
    ax.add_collection3d(make_edge_glow_layer(pulse))
    ax.add_collection3d(make_edge_core_layer(pulse))

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


# -----------------------------------------------------------------------------
# Export
# -----------------------------------------------------------------------------

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="schwarz_p_surface_v2_grid",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/schwarz_p_surface_v2_grid.webm
Frames: 192
Size: 14860.3 KB


[out#0/webm @ 0x12ff042e0] video:14858KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.013710%
frame=  192 fps= 13 q=32.0 Lsize=   14860KiB time=00:00:08.00 bitrate=15217.0kbits/s speed=0.54x    


The Schwarz P surface is a triply periodic minimal surface defined by the implicit equation:

$$ \cos(x) + \cos(y) + \cos(z) = 0 $$

It repeats in three spatial directions and locally minimizes area, like a soap film stretched through a periodic lattice. Visually, it forms a smooth cubic labyrinth, making it useful in geometry, materials science, porous media, crystallography, and architectural mathematics.

# 3. Boy Surface

In [13]:
# Boy Surface — rotating mathematical sculpture
# Output: media-site/animations/Math/boy_surface_v2_grid.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W, H = 10, 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"

COL = "#35f6ff"
COL_GRID_U = "white"
COL_GRID_V = "#9ffcff"

SURFACE_ALPHA = 0.18
GRID_GLOW_ALPHA = 0.65
GRID_CORE_ALPHA = 0.98

GRID_GLOW_WIDTH = 1.35
GRID_CORE_WIDTH = 0.48

GRID_STEP_U = 8
GRID_STEP_V = 8


# -----------------------------------------------------------------------------
# Boy surface geometry
# Parametric immersion of the real projective plane into 3D.
# -----------------------------------------------------------------------------

u = np.linspace(0.001, np.pi - 0.001, 180)
v = np.linspace(0.0, 2 * np.pi, 240)

U, V = np.meshgrid(u, v)

sqrt2 = np.sqrt(2.0)

den = 2.0 - sqrt2 * np.sin(3 * U) * np.sin(2 * V)

X = (
    sqrt2 * np.cos(2 * U) * np.cos(2 * V)
    + np.cos(U) * np.cos(V)
) / den

Y = (
    sqrt2 * np.cos(2 * U) * np.sin(2 * V)
    - np.cos(U) * np.sin(V)
) / den

Z = (3.0 * np.cos(U)) / den

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.45
Y = Y / scale * 3.45
Z = Z / scale * 3.45


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-2.7, 2.7)
    ax.set_ylim(-2.7, 2.7)
    ax.set_zlim(-2.7, 2.7)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    # U-lines
    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    # V-lines
    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 7 * np.sin(tau * t),
        azim=360 * t,
    )

    colors = build_colors(pulse)

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=colors,
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


# -----------------------------------------------------------------------------
# Export
# -----------------------------------------------------------------------------

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="boy_surface_v2_grid",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/boy_surface_v2_grid.webm
Frames: 192
Size: 2573.4 KB


The Boy surface is an immersion of the real projective plane into three-dimensional space. It is a non-orientable surface, like the Möbius strip and Klein bottle, but it has no boundary. Since the real projective plane cannot be embedded in ordinary 3D space without self-intersections, the Boy surface shows it as a smooth self-intersecting form. Visually, it looks organic and alien, which makes it useful for this mathematical sculpture series.

$$
x =
\frac{
\sqrt{2}\cos(2u)\cos(2v)+\cos u\cos v
}{
2-\sqrt{2}\sin(3u)\sin(2v)
}

$$

$$
y =
\frac{
\sqrt{2}\cos(2u)\sin(2v)-\cos u\sin v
}{
2-\sqrt{2}\sin(3u)\sin(2v)
}

$$

$$
z =
\frac{
3\cos u
}{
2-\sqrt{2}\sin(3u)\sin(2v)
}
$$

# Roman Surface

In [11]:
# Roman Surface — rotating mathematical sculpture
# Output: media-site/animations/Math/roman_surface_v1.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W, H = 10, 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"

COL = "#35f6ff"
COL_GRID_U = "white"
COL_GRID_V = "#9ffcff"

SURFACE_ALPHA = 0.18
GRID_GLOW_ALPHA = 0.65
GRID_CORE_ALPHA = 0.98

GRID_GLOW_WIDTH = 1.35
GRID_CORE_WIDTH = 0.48

GRID_STEP_U = 8
GRID_STEP_V = 8


# -----------------------------------------------------------------------------
# Roman surface geometry
#
# Parametric form:
#   x = sin(2u) * sin^2(v)
#   y = sin(u) * sin(2v)
#   z = cos(u) * sin(2v)
#
# It is an immersion of the real projective plane into 3D,
# with characteristic self-intersections.
# -----------------------------------------------------------------------------

u = np.linspace(0.0, np.pi, 180)
v = np.linspace(0.0, np.pi, 180)

U, V = np.meshgrid(u, v)

X = np.sin(2 * U) * np.sin(V) ** 2
Y = np.sin(U) * np.sin(2 * V)
Z = np.cos(U) * np.sin(2 * V)

scale = np.max(np.abs([X, Y, Z]))
X = X / scale * 3.25
Y = Y / scale * 3.25
Z = Z / scale * 3.25


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    # U-lines
    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    # V-lines
    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 7 * np.sin(tau * t),
        azim=360 * t,
    )

    colors = build_colors(pulse)

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=colors,
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


# -----------------------------------------------------------------------------
# Export
# -----------------------------------------------------------------------------

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="roman_surface_v1",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/roman_surface_v1.webm
Frames: 192
Size: 3588.5 KB


[out#0/webm @ 0x145e245d0] video:3587KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.053858%
frame=  192 fps= 36 q=32.0 Lsize=    3588KiB time=00:00:08.00 bitrate=3674.6kbits/s speed= 1.5x    


The Roman surface is a self-intersecting immersion of the real projective plane into three-dimensional space. It has a characteristic threefold symmetry and three intersecting “sheets”, which makes it look like a mathematical flower or topological artifact. Like the Boy surface, it cannot represent the projective plane as a clean embedded surface in ordinary 3D space, so self-intersection is part of the geometry.

# Dini Surface

In [22]:
# Dini Surface — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/dini_surface_v1_grid.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W, H = 10, 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"

COL = "#35f6ff"
COL_GRID_U = "white"
COL_GRID_V = "#9ffcff"

SURFACE_ALPHA = 0.16
GRID_GLOW_ALPHA = 0.65
GRID_CORE_ALPHA = 0.98

GRID_GLOW_WIDTH = 1.35
GRID_CORE_WIDTH = 0.48

GRID_STEP_U = 6
GRID_STEP_V = 8


# -----------------------------------------------------------------------------
# Dini surface geometry
#
# Parametric form:
#   x = a cos(u) sin(v)
#   y = a sin(u) sin(v)
#   z = a (cos(v) + log(tan(v/2))) + b u
#
# It is a helicoidal surface related to pseudospherical geometry.
# -----------------------------------------------------------------------------

a = 1.0
b = 0.18

u = np.linspace(0.0, 5.4 * np.pi, 260)
v = np.linspace(0.16, 2.45, 140)

U, V = np.meshgrid(u, v)

X = a * np.cos(U) * np.sin(V)
Y = a * np.sin(U) * np.sin(V)
Z = a * (np.cos(V) + np.log(np.tan(V / 2.0))) + b * U

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.0
Y = Y / scale * 4.0
Z = Z / scale * 4.0
Z -= 1.6


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.8, 1.8)
    ax.set_zlim(-1.8, 1.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    # U-lines
    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    # V-lines
    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=27 + 7 * np.sin(tau * t),
        azim=360 * t,
    )

    colors = build_colors(pulse)

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=colors,
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


# -----------------------------------------------------------------------------
# Export
# -----------------------------------------------------------------------------

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="dini_surface_v1_grid",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/dini_surface_v1_grid.webm
Frames: 192
Size: 3858.6 KB


[out#0/webm @ 0x136e041c0] video:3857KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.050441%
frame=  192 fps= 35 q=32.0 Lsize=    3859KiB time=00:00:08.00 bitrate=3951.2kbits/s speed=1.44x    


Dini’s surface is a twisted surface related to constant negative curvature. Visually it resembles a spiral shell or helicoidal membrane, making it a strong candidate for a mathematical “alien artifact” style animation.

# Helicoid

In [23]:
# Helicoid — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/helicoid_v1_grid.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W, H = 10, 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"

COL = "#35f6ff"
COL_GRID_U = "white"
COL_GRID_V = "#9ffcff"

SURFACE_ALPHA = 0.14
GRID_GLOW_ALPHA = 0.65
GRID_CORE_ALPHA = 0.98

GRID_GLOW_WIDTH = 1.35
GRID_CORE_WIDTH = 0.48

GRID_STEP_U = 7
GRID_STEP_V = 8


# -----------------------------------------------------------------------------
# Helicoid geometry
#
#   x = u cos(v)
#   y = u sin(v)
#   z = c v
# -----------------------------------------------------------------------------

u = np.linspace(-2.2, 2.2, 150)
v = np.linspace(-2.8 * np.pi, 2.8 * np.pi, 260)

U, V = np.meshgrid(u, v)

c = 0.34

X = U * np.cos(V)
Y = U * np.sin(V)
Z = c * V

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.2
Y = Y / scale * 4.2
Z = Z / scale * 4.2


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    # U-lines
    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    # V-lines
    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=26 + 7 * np.sin(tau * t),
        azim=360 * t,
    )

    colors = build_colors(pulse)

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=colors,
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


# -----------------------------------------------------------------------------
# Export
# -----------------------------------------------------------------------------

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name="helicoid_v1_grid",
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/helicoid_v1_grid.webm
Frames: 192
Size: 5627.7 KB


[out#0/webm @ 0x153f1bb60] video:5626KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.036176%
frame=  192 fps= 24 q=32.0 Lsize=    5628KiB time=00:00:08.00 bitrate=5762.8kbits/s speed=   1x    


The helicoid is a classical minimal surface generated by a straight line rotating around an axis while moving upward. It is one of the simplest non-planar minimal surfaces and is closely related to the catenoid. Visually it resembles a twisted membrane, screw surface, or mathematical spiral ramp.

# Enneper Surface

In [26]:
# Enneper Surface — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/enneper_surface_v1_grid.webm

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "enneper_surface_v1_grid"


# -----------------------------------------------------------------------------
# Enneper surface geometry
#
# x = u - u^3/3 + u v^2
# y = v - v^3/3 + v u^2
# z = u^2 - v^2
# -----------------------------------------------------------------------------

u = np.linspace(-1.9, 1.9, 180)
v = np.linspace(-1.9, 1.9, 180)

U, V = np.meshgrid(u, v)

X = U - (U**3) / 3.0 + U * V**2
Y = V - (V**3) / 3.0 + V * U**2
Z = U**2 - V**2

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.85
Y = Y / scale * 3.85
Z = Z / scale * 3.85


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=30,
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/enneper_surface_v1_grid.webm
Frames: 192
Size: 2844.4 KB


The Enneper surface is a classical minimal surface with self-intersections. It is generated from a simple polynomial parametrization, but its shape forms a complex flared structure, often resembling a mathematical flower, coral, or folded membrane.

# Catenoid

In [27]:
# Catenoid — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/catenoid_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "catenoid_v1_grid"


# -----------------------------------------------------------------------------
# Catenoid geometry
#
# x = a cosh(v) cos(u)
# y = a cosh(v) sin(u)
# z = a v
# -----------------------------------------------------------------------------

u = np.linspace(0, 2 * np.pi, 220)
v = np.linspace(-1.55, 1.55, 150)

U, V = np.meshgrid(u, v)

a = 1.0

X = a * np.cosh(V) * np.cos(U)
Y = a * np.cosh(V) * np.sin(U)
Z = a * V

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.75
Y = Y / scale * 3.75
Z = Z / scale * 3.75


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=25 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/catenoid_v1_grid.webm
Frames: 192
Size: 2938.2 KB


The catenoid is a classical minimal surface formed by rotating a catenary curve around an axis. It is the only minimal surface of revolution besides the plane, and it resembles a smooth neck or mathematical wormhole between two circular rims.

$ x=a\cosh(v)\cos(u), $
$ y=a\cosh(v)\sin(u), $
$ z=a v $

где:

* $u \in [0,2\pi]$ — угол вокруг оси;
* $v$ — координата вдоль поверхности;
* $a$ — масштаб.

# Whitney Umbrella

In [28]:
# Whitney Umbrella — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/whitney_umbrella_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "whitney_umbrella_v1_grid"


# -----------------------------------------------------------------------------
# Whitney umbrella geometry
#
# x = u v
# y = u
# z = v^2
# -----------------------------------------------------------------------------

u = np.linspace(-2.2, 2.2, 180)
v = np.linspace(-2.2, 2.2, 180)

U, V = np.meshgrid(u, v)

X = U * V
Y = U
Z = V**2

# Center Z around origin for better rotation
Z = Z - Z.mean()

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.85
Y = Y / scale * 3.85
Z = Z / scale * 3.85


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=26 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/whitney_umbrella_v1_grid.webm
Frames: 192
Size: 2461.6 KB


[out#0/webm @ 0x1396122a0] video:2460KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.076665%
frame=  192 fps= 41 q=32.0 Lsize=    2462KiB time=00:00:08.00 bitrate=2520.7kbits/s speed=1.73x    


The Whitney umbrella is a classical singular surface. Its parametrization is simple, but it creates a self-intersection line and a pinch-point singularity. Visually, it looks like a folded mathematical sheet passing through itself, which makes it especially effective in wireframe form.

$$ x=uv, y=u, z=v^2 $$

# Monkey Saddle

In [40]:
# Monkey Saddle — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/monkey_saddle_v2_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "monkey_saddle_v2_grid"


X_STRETCH = 3.0
Y_STRETCH = 2.0
Z_STRETCH = 0.46

BASE_SCALE = 3.0

CAMERA_X = 6.4
CAMERA_Y = 4.2
CAMERA_Z = 2.4



# -----------------------------------------------------------------------------
# Monkey Saddle geometry
#
# z = r^3 cos(3θ)
# -----------------------------------------------------------------------------

r = np.linspace(0.0, 1.8, 180)
theta = np.linspace(0.0, 2 * np.pi, 240)

R, T = np.meshgrid(r, theta)

X = R * np.cos(T)
Y = R * np.sin(T)

Z = 0.45 * R**3 * np.cos(3 * T)

# optional artistic tuning
X *= 1.3
Y *= 1.3

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.8
Y = Y / scale * 3.8
Z = Z / scale * 3.8



# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-4.2, 4.2)
    ax.set_ylim(-4.2, 4.2)
    ax.set_zlim(-2.8, 2.8)

    ax.set_box_aspect((1, 1, 0.7))

    ax.grid(False)
    ax.set_axis_off()

def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(X[i, :], Y[i, :], Z[i, :], color=COL,
                linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[i, :], Y[i, :], Z[i, :], color=COL_GRID_U,
                linewidth=GRID_CORE_WIDTH, alpha=core_alpha)

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(X[:, j], Y[:, j], Z[:, j], color=COL,
                linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[:, j], Y[:, j], Z[:, j], color=COL_GRID_V,
                linewidth=GRID_CORE_WIDTH, alpha=core_alpha)


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=32,
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/monkey_saddle_v2_grid.webm
Frames: 192
Size: 3479.5 KB


[out#0/webm @ 0x14d715100] video:3478KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.055237%
frame=  192 fps= 35 q=32.0 Lsize=    3480KiB time=00:00:08.00 bitrate=3563.0kbits/s speed=1.47x    


The monkey saddle is a classical saddle surface with threefold symmetry. Its equation is simple, but it creates a distinctive three-pronged shape that visually resembles a monkey gripping a branch with two arms and a tail.

$$ z = x^3 - 3xy^2 $$

The monkey saddle is a saddle surface with three downward directions instead of the usual two. Its threefold symmetry makes it look like a mathematical flower or a warped tri-radial membrane.

# Möbius Strip

In [41]:
# Möbius Strip — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/mobius_strip_v2_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "mobius_strip_v2_grid"


# -----------------------------------------------------------------------------
# Möbius strip geometry
#
# x = (R + v cos(u/2)) cos(u)
# y = (R + v cos(u/2)) sin(u)
# z = v sin(u/2)
#
# One-sided surface.
# -----------------------------------------------------------------------------

u = np.linspace(0, 2 * np.pi, 260)
v = np.linspace(-0.85, 0.85, 90)

U, V = np.meshgrid(u, v)

R = 2.1

X = (R + V * np.cos(U / 2)) * np.cos(U)
Y = (R + V * np.cos(U / 2)) * np.sin(U)
Z = V * np.sin(U / 2)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.2
Y = Y / scale * 4.2
Z = Z / scale * 4.2


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-4.2, 4.2)
    ax.set_ylim(-4.2, 4.2)
    ax.set_zlim(-4.2, 4.2)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(
        int(hex_color[i:i + 2], 16) / 255
        for i in (0, 2, 4)
    )


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))

    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:

    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    # U-lines
    for i in range(0, X.shape[0], GRID_STEP_U):

        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )

        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    # V-lines
    for j in range(0, X.shape[1], GRID_STEP_V):

        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )

        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []

tau = 2 * np.pi

for frame_idx in range(FRAMES):

    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)

    setup_axes()

    ax.view_init(
        elev=28,
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


# -----------------------------------------------------------------------------
# Export
# -----------------------------------------------------------------------------

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/mobius_strip_v2_grid.webm
Frames: 192
Size: 1576.3 KB


Интересный факт про ленту Мёбиуса: если начать двигаться по поверхности, не переходя через край, то после одного полного оборота вы окажетесь на «другой стороне», а после второго — вернётесь в исходную точку. У неё только одна сторона и только один край. Именно эта идея затем приводит к бутылке Клейна, где даже край исчезает полностью.

# Trefoil Knot Tube

In [42]:
# Trefoil Knot Tube — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/trefoil_knot_tube_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "trefoil_knot_tube_v1_grid"


# -----------------------------------------------------------------------------
# Trefoil knot tube geometry
# -----------------------------------------------------------------------------

n_path = 360
n_tube = 48

t_path = np.linspace(0, 2 * np.pi, n_path)
theta = np.linspace(0, 2 * np.pi, n_tube)

T, TH = np.meshgrid(t_path, theta)

# Trefoil centerline
cx = np.sin(t_path) + 2 * np.sin(2 * t_path)
cy = np.cos(t_path) - 2 * np.cos(2 * t_path)
cz = -np.sin(3 * t_path)

C = np.vstack([cx, cy, cz]).T

# Tangent vectors
dC = np.gradient(C, axis=0)
Tangent = dC / np.linalg.norm(dC, axis=1, keepdims=True)

# Build stable normal/binormal frame
up = np.array([0.0, 0.0, 1.0])
Normals = np.cross(Tangent, up)

bad = np.linalg.norm(Normals, axis=1) < 1e-6
Normals[bad] = np.cross(Tangent[bad], np.array([0.0, 1.0, 0.0]))

Normals = Normals / np.linalg.norm(Normals, axis=1, keepdims=True)
Binormals = np.cross(Tangent, Normals)
Binormals = Binormals / np.linalg.norm(Binormals, axis=1, keepdims=True)

tube_r = 0.18

X = np.zeros((n_tube, n_path))
Y = np.zeros((n_tube, n_path))
Z = np.zeros((n_tube, n_path))

for i in range(n_path):
    for j in range(n_tube):
        offset = (
            tube_r * np.cos(theta[j]) * Normals[i]
            + tube_r * np.sin(theta[j]) * Binormals[i]
        )

        p = C[i] + offset

        X[j, i] = p[0]
        Y[j, i] = p[1]
        Z[j, i] = p[2]

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.0
Y = Y / scale * 4.0
Z = Z / scale * 4.0


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/trefoil_knot_tube_v1_grid.webm
Frames: 192
Size: 1962.8 KB


[out#0/webm @ 0x13b80ac20] video:1961KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.095969%
frame=  192 fps= 55 q=32.0 Lsize=    1963KiB time=00:00:08.00 bitrate=2009.9kbits/s speed=2.29x    


Trefoil knot — простейший нетривиальный узел. В отличие от окружности, его нельзя распутать в плоское кольцо без разрезания.

# Seifert Surface

In [43]:
# Seifert Surface — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/seifert_surface_trefoil_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "seifert_surface_trefoil_v1_grid"


# -----------------------------------------------------------------------------
# Seifert-like surface for a trefoil boundary
#
# Trefoil boundary:
# x = sin(t) + 2 sin(2t)
# y = cos(t) - 2 cos(2t)
# z = -sin(3t)
#
# Surface is built by interpolating radial layers from a central core
# to the trefoil boundary.
# -----------------------------------------------------------------------------

n_path = 360
n_radial = 90

t_path = np.linspace(0, 2 * np.pi, n_path)
s_radial = np.linspace(0.02, 1.0, n_radial)

T, S = np.meshgrid(t_path, s_radial)

BX = np.sin(T) + 2 * np.sin(2 * T)
BY = np.cos(T) - 2 * np.cos(2 * T)
BZ = -np.sin(3 * T)

# Twisted central spine to avoid a flat cone-like fill
CX = 0.25 * np.sin(3 * T)
CY = 0.25 * np.cos(3 * T)
CZ = 0.18 * np.sin(2 * T)

# Nonlinear radial growth makes the membrane more organic
S2 = S**0.72

X = (1 - S2) * CX + S2 * BX
Y = (1 - S2) * CY + S2 * BY
Z = (1 - S2) * CZ + S2 * BZ

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.0
Y = Y / scale * 4.0
Z = Z / scale * 4.0


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    # Radial bands
    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    # Boundary-direction bands
    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


def draw_trefoil_boundary(pulse: float) -> None:
    bx = X[-1, :]
    by = Y[-1, :]
    bz = Z[-1, :]

    ax.plot(
        bx,
        by,
        bz,
        color=COL,
        linewidth=3.2,
        alpha=0.28 + 0.22 * pulse,
    )

    ax.plot(
        bx,
        by,
        bz,
        color="white",
        linewidth=0.9,
        alpha=0.75 + 0.2 * pulse,
    )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)
    draw_trefoil_boundary(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/seifert_surface_trefoil_v1_grid.webm
Frames: 192
Size: 2254.3 KB


[out#0/webm @ 0x12ef23840] video:2252KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.083589%
frame=  192 fps= 41 q=32.0 Lsize=    2254KiB time=00:00:08.00 bitrate=2308.4kbits/s speed=1.71x    


# Dupin Cyclide

In [44]:
# Dupin Cyclide — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/dupin_cyclide_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "dupin_cyclide_v1_grid"


# -----------------------------------------------------------------------------
# Dupin cyclide geometry
# -----------------------------------------------------------------------------

u = np.linspace(0, 2 * np.pi, 240)
v = np.linspace(0, 2 * np.pi, 160)

U, V = np.meshgrid(u, v)

a = 2.2
b = 1.0
c = 0.85

den = a - c * np.cos(U) * np.cos(V)

X = (b * np.cos(U) * (a - c * np.cos(V))) / den
Y = (b * np.sin(U) * (a - c * np.cos(V))) / den
Z = (b * np.sin(V) * (a - c * np.cos(U))) / den

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.0
Y = Y / scale * 4.0
Z = Z / scale * 4.0


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()
    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/dupin_cyclide_v1_grid.webm
Frames: 192
Size: 3429.8 KB


[out#0/webm @ 0x131e24030] video:3428KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.056408%
frame=  192 fps= 40 q=32.0 Lsize=    3430KiB time=00:00:08.00 bitrate=3512.1kbits/s speed=1.66x    


# Kuen surface 

In [45]:
# Kuen Surface — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/kuen_surface_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "kuen_surface_v1_grid"


# -----------------------------------------------------------------------------
# Kuen surface geometry
# -----------------------------------------------------------------------------

u = np.linspace(-4.2, 4.2, 220)
v = np.linspace(0.08, np.pi - 0.08, 160)

U, V = np.meshgrid(u, v)

den = 1.0 + U**2 * np.sin(V) ** 2

X = (2.0 * (np.cos(U) + U * np.sin(U)) * np.sin(V)) / den
Y = (2.0 * (np.sin(U) - U * np.cos(U)) * np.sin(V)) / den
Z = np.log(np.tan(V / 2.0)) + (2.0 * np.cos(V)) / den

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.0
Y = Y / scale * 4.0
Z = Z / scale * 4.0


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()
    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/kuen_surface_v1_grid.webm
Frames: 192
Size: 2866.2 KB


[out#0/webm @ 0x14a904f60] video:2864KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.066415%
frame=  192 fps= 41 q=32.0 Lsize=    2866KiB time=00:00:08.00 bitrate=2935.0kbits/s speed=1.71x    


Kuen surface is a classical surface of constant negative curvature. It is closely related to pseudospherical geometry and has a sharp, twisted, vortex-like shape.

# Plücker Conoid

In [46]:
# Plücker Conoid — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/plucker_conoid_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "plucker_conoid_v1_grid"


# -----------------------------------------------------------------------------
# Plücker conoid geometry
#
# x = u cos(v)
# y = u sin(v)
# z = sin(n v)
# -----------------------------------------------------------------------------

u = np.linspace(-2.2, 2.2, 170)
v = np.linspace(0.0, 2 * np.pi, 260)

U, V = np.meshgrid(u, v)

N_FOLD = 3

X = U * np.cos(V)
Y = U * np.sin(V)
Z = 1.25 * np.sin(N_FOLD * V)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.0
Y = Y / scale * 4.0
Z = Z / scale * 4.0


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()
    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/plucker_conoid_v1_grid.webm
Frames: 192
Size: 5898.4 KB


[out#0/webm @ 0x12b804520] video:5896KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.034516%
frame=  192 fps= 24 q=32.0 Lsize=    5898KiB time=00:00:08.00 bitrate=6040.0kbits/s speed=0.988x    


# Steiner Surface

In [47]:
# Steiner Surface — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/steiner_surface_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "steiner_surface_v1_grid"


# -----------------------------------------------------------------------------
# Steiner surface geometry
#
# One common parametrization:
# x = sin(2u) cos^2(v)
# y = sin(u) sin(2v)
# z = cos(u) sin(2v)
# -----------------------------------------------------------------------------

u = np.linspace(0.0, np.pi, 190)
v = np.linspace(0.0, np.pi, 190)

U, V = np.meshgrid(u, v)

X = np.sin(2 * U) * np.cos(V) ** 2
Y = np.sin(U) * np.sin(2 * V)
Z = np.cos(U) * np.sin(2 * V)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.75
Y = Y / scale * 3.75
Z = Z / scale * 3.75


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()
    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/steiner_surface_v1_grid.webm
Frames: 192
Size: 5016.7 KB


Steiner surface is another self-intersecting model of the real projective plane. In this style it should look like a topological crystal or folded mathematical flower.

# Bour Surface

In [48]:
# Bour Surface — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/bour_surface_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "bour_surface_v1_grid"


# -----------------------------------------------------------------------------
# Bour surface geometry
# -----------------------------------------------------------------------------

u = np.linspace(0.18, 1.85, 180)
v = np.linspace(0.0, 2 * np.pi, 240)

U, V = np.meshgrid(u, v)

X = U * np.cos(V) - (U**3 / 3.0) * np.cos(3 * V)
Y = -U * np.sin(V) - (U**3 / 3.0) * np.sin(3 * V)
Z = U**2 * np.cos(2 * V)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.9
Y = Y / scale * 3.9
Z = Z / scale * 3.9


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.7, 3.7)
    ax.set_ylim(-3.7, 3.7)
    ax.set_zlim(-3.7, 3.7)
    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)
    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(X[i, :], Y[i, :], Z[i, :], color=COL,
                linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[i, :], Y[i, :], Z[i, :], color=COL_GRID_U,
                linewidth=GRID_CORE_WIDTH, alpha=core_alpha)

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(X[:, j], Y[:, j], Z[:, j], color=COL,
                linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[:, j], Y[:, j], Z[:, j], color=COL_GRID_V,
                linewidth=GRID_CORE_WIDTH, alpha=core_alpha)


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()
    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/bour_surface_v1_grid.webm
Frames: 192
Size: 4066.4 KB


[out#0/webm @ 0x149e1dbe0] video:4064KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.048174%
frame=  192 fps= 29 q=32.0 Lsize=    4066KiB time=00:00:08.00 bitrate=4164.0kbits/s speed=1.21x    


Bour surface is a minimal surface with strong rotational structure. In wireframe form it reads as a twisted organic shell or topological sea creature.

# Astroidal Ellipsoid

In [50]:
# Astroidal Ellipsoid — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/astroidal_ellipsoid_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "astroidal_ellipsoid_v1_grid"


# -----------------------------------------------------------------------------
# Astroidal ellipsoid geometry
#
# x = a cos^3(u) cos^3(v)
# y = b cos^3(u) sin^3(v)
# z = c sin^3(u)
# -----------------------------------------------------------------------------

u = np.linspace(-np.pi / 2, np.pi / 2, 180)
v = np.linspace(0.0, 2 * np.pi, 240)

U, V = np.meshgrid(u, v)

a = 1.0
b = 1.0
c = 1.15

X = a * np.cos(U) ** 3 * np.cos(V) ** 3
Y = b * np.cos(U) ** 3 * np.sin(V) ** 3
Z = c * np.sin(U) ** 3

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.85
Y = Y / scale * 3.85
Z = Z / scale * 3.85


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/astroidal_ellipsoid_v1_grid.webm
Frames: 192
Size: 2246.7 KB


[out#0/webm @ 0x137904820] video:2245KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.083875%
frame=  192 fps= 47 q=32.0 Lsize=    2247KiB time=00:00:08.00 bitrate=2300.6kbits/s speed=1.96x    


$$ x = a \cos^3(u)\cos^3(v)$$

$$y = b \cos^3(u)\sin^3(v)$$

$$z = c \sin^3(u)$$

It should look like a smooth mathematical crystal or organic capsule.

# Breather Surface

In [51]:
# Breather Surface — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/breather_surface_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "breather_surface_v1_grid"


# -----------------------------------------------------------------------------
# Breather surface geometry
#
# A classical pseudospherical breather-like parametrization.
# -----------------------------------------------------------------------------

u = np.linspace(-13.0, 13.0, 240)
v = np.linspace(-1.35, 1.35, 140)

U, V = np.meshgrid(u, v)

a = 0.45
b = np.sqrt(1.0 - a**2)

den = a * ((b * np.cosh(a * U)) ** 2 + (a * np.sin(b * V)) ** 2)

X = -U + (2.0 * (1.0 - a**2) * np.cosh(a * U) * np.sinh(a * U)) / den
Y = (2.0 * b * np.cosh(a * U) * (-(b * np.cos(V) * np.cos(b * V)) - np.sin(V) * np.sin(b * V))) / den
Z = (2.0 * b * np.cosh(a * U) * (-(b * np.sin(V) * np.cos(b * V)) + np.cos(V) * np.sin(b * V))) / den

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.1
Y = Y / scale * 4.1
Z = Z / scale * 4.1


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/breather_surface_v1_grid.webm
Frames: 192
Size: 232.3 KB


[out#0/webm @ 0x12e907480] video:230KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.800949%
frame=  192 fps= 93 q=32.0 Lsize=     232KiB time=00:00:08.00 bitrate= 237.9kbits/s speed=3.88x    


Breather Surface — волновая поверхность из геометрии постоянной отрицательной кривизны. Визуально должна выглядеть как застывшая солитонная волна или скрученная мембрана.

# Catalan Surface 

In [52]:
# Catalan Surface — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/catalan_surface_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "catalan_surface_v1_grid"


# -----------------------------------------------------------------------------
# Catalan surface geometry
#
# x = u - sin(u) cosh(v)
# y = 1 - cos(u) cosh(v)
# z = 4 sin(u/2) sinh(v/2)
# -----------------------------------------------------------------------------

u = np.linspace(-np.pi, np.pi, 220)
v = np.linspace(-1.65, 1.65, 150)

U, V = np.meshgrid(u, v)

X = U - np.sin(U) * np.cosh(V)
Y = 1.0 - np.cos(U) * np.cosh(V)
Z = 4.0 * np.sin(U / 2.0) * np.sinh(V / 2.0)

# Center
X -= X.mean()
Y -= Y.mean()
Z -= Z.mean()

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.0
Y = Y / scale * 4.0
Z = Z / scale * 4.0


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=27 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/catalan_surface_v1_grid.webm
Frames: 192
Size: 3350.0 KB


Catalan Surface — минимальная поверхность с волнистой симметрией. В нашем “проволочном музейном” стиле должна выглядеть как плавная математическая мембрана, менее агрессивная, чем Kuen/Bour, но очень чистая геометрически.

$$ x=u-\sin(u)\cosh(v) $$

$$ y=1-\cos(u)\cosh(v) $$

$$ z=4\sin(u/2)\sinh(v/2) $$

Catalan Surface is a classical minimal surface with a wave-like folded structure.

# Scherk Surface

In [53]:
# Scherk Surface — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/scherk_surface_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "scherk_surface_v1_grid"


# -----------------------------------------------------------------------------
# Scherk surface geometry
#
# z = log(cos(y) / cos(x))
#
# Domain is restricted to avoid singularities where cos(x) or cos(y) = 0.
# -----------------------------------------------------------------------------

u = np.linspace(-1.34, 1.34, 190)
v = np.linspace(-1.34, 1.34, 190)

U, V = np.meshgrid(u, v)

X = U
Y = V
Z = np.log(np.cos(V) / np.cos(U))

# Artistic compression: keeps vertical growth readable
Z *= 0.75

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.85
Y = Y / scale * 3.85
Z = Z / scale * 3.85


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.6, 3.6)
    ax.set_ylim(-3.6, 3.6)
    ax.set_zlim(-3.6, 3.6)

    ax.set_box_aspect((1, 1, 0.85))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=30 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/scherk_surface_v1_grid.webm
Frames: 192
Size: 3901.3 KB


[out#0/webm @ 0x12a61ebb0] video:3899KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.049763%
frame=  192 fps= 30 q=32.0 Lsize=    3901KiB time=00:00:08.00 bitrate=3994.9kbits/s speed=1.25x    


Formula:

$$ z=\log\left(\frac{\cos y}{\cos x}\right) $$

Scherk Surface is a classical minimal surface. It has saddle-like passages and vertical growth near its singular boundaries, which gives it an architectural, bridge-like structure.

# Maeder’s Owl Surface

In [54]:
# Maeder's Owl Surface — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/maeders_owl_surface_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "maeders_owl_surface_v1_grid"


# -----------------------------------------------------------------------------
# Maeder's Owl surface geometry
#
# x = v cos(u) - 0.5 v^2 cos(2u)
# y = -v sin(u) - 0.5 v^2 sin(2u)
# z = 4 v^1.5 cos(3u/2) / 3
# -----------------------------------------------------------------------------

u = np.linspace(0.0, 4.0 * np.pi, 260)
v = np.linspace(0.0, 1.45, 150)

U, V = np.meshgrid(u, v)

X = V * np.cos(U) - 0.5 * V**2 * np.cos(2 * U)
Y = -V * np.sin(U) - 0.5 * V**2 * np.sin(2 * U)
Z = (4.0 / 3.0) * V**1.5 * np.cos(1.5 * U)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.9
Y = Y / scale * 3.9
Z = Z / scale * 3.9


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.7, 3.7)
    ax.set_ylim(-3.7, 3.7)
    ax.set_zlim(-3.7, 3.7)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/maeders_owl_surface_v1_grid.webm
Frames: 192
Size: 4334.2 KB


[out#0/webm @ 0x157e0db30] video:4332KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.045602%
frame=  192 fps= 30 q=32.0 Lsize=    4334KiB time=00:00:08.00 bitrate=4438.3kbits/s speed=1.25x    


# Seashell Surface

In [55]:
# Seashell Surface — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/seashell_surface_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "seashell_surface_v1_grid"


# -----------------------------------------------------------------------------
# Seashell surface geometry
#
# Logarithmic spiral shell-like parametric surface.
# -----------------------------------------------------------------------------

u = np.linspace(0.0, 5.5 * np.pi, 260)
v = np.linspace(-0.75, 0.75, 120)

U, V = np.meshgrid(u, v)

A = 0.16
B = 0.13
C = 0.95

radius = A * np.exp(B * U)

tube = 1.0 + 0.38 * np.cos(V)

X = radius * tube * np.cos(U)
Y = radius * tube * np.sin(U)
Z = C * radius * np.sin(V) + 0.10 * U

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.0
Y = Y / scale * 4.0
Z = Z / scale * 4.0


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str) -> tuple[float, float, float]:
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float) -> np.ndarray:
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/seashell_surface_v1_grid.webm
Frames: 192
Size: 1436.2 KB


[out#0/webm @ 0x12cf25830] video:1434KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.128883%
frame=  192 fps= 56 q=32.0 Lsize=    1436KiB time=00:00:08.00 bitrate=1470.7kbits/s speed=2.33x    
